# This Notebook is for finding the oil images with red circle from the ones that don't have a red circle

Oil annotation guidelines
- bounding around whole oil spillages. Unless it covers the entire image and overlaps too much white tank parts. 
- If you don't what to do, leave it open.
- If you're not sure about an oil stain, tag the image as possible-oil-stain


todo
- check minimum pixel size for detection in RF-DETR and YOLO models -> 1280x1280 to detect 20px minimum spillage
- check if stains on support-legs are clearly visible in lowest resolution (verify with above minimal pixel size)




Pilot beschrijving

Voor de samenwerking tussen IndiVillage en Falcker is het belangrijk dat we een goede basis leggen voor een langdurige samenwerking.
Indivillage als partij gaat helpen bij het annoteren van data en produceren van modellen. Bij het verzamelen en samenstellen van data kunnen ze ondersteunen, indien blijkt dat er meer data nodig is of er wijzigingen in data moeten zijn. Falcker zal hierin proberen te voorzien.

Als eerste project zullen we kijken naar het detecteren van product lekkages op External FLoating Roof daken.
Hierbinnen zijn verschillende scenario's waarbij we als eerst zullen kijken naar spillage uit de seal op en bij de pontoons.


Probleembeschrijving
Onze klanten hebben grote opslag tanks die onder andere CRUD olie bevatten. Deze olie kan via verschillende openingen in en op de tank ontsnappen. Bij het ontsnappen van de olie is het van belang dat deze zsm gedetecteerd en opgeruimd wordt. Wij kijken specifiek naar het type EFR tank wat drijvende daken heeft waarbij er product kan lekken via openingen op het dak zelf en aan de zijkant van het dak bij de seal. Bij gasophoping of ander falen van operationele processen kan er product via de seal op het dak terecht komen. 

Er zijn verschillende type product lekkages, we zullen alleen kijken naar de lekkages die bij de seal voorkomen. Dit zijn verfspetter achtige patronen die uit kunnen lopen in stromen. Deze beginnen aan de rand van de tank, op de seal over de pontoon en dan bij een verdere uitloop ook op het centre-deck. Soms zijn er simpele vlekken, maar het zal anders zijn dan product lekkages bij uitstekende open verbindingen bovenop het tank dak zoals Support-legs of bleeder vents. Van dit soort seal product lekkages hebben we rond de 100 foto's met product en een veelvoud (900) aan foto's zonder product lekkages.


Gewenst Resultaat:
Uiteindelijk willen we een detectie model met bounding boxen waarbij we enkel één klasse detecteren: product.
Product lekkage is ernstig en gevaarlijk en daarom is het belangrijk dat we zo min mogelijk lekkages missen en dus een hoge recall hebben. 
Bij een detectie is het minder van belang dat de detectie zelf exact de goede plek zit, dus precision is van minder belang, maar nog steeds belangrijk.
Er zijn verschillende manieren om AI modellen te toetsen. De meest gebruikte methode is de mAP (mean Average Precision) bij een overlap van 50%(mAP@.5). Deze is gebasseerd op de F1 score, waarbij beide precision en recall gelijkwaardig worden meegenomen. Naast de F1 score is er ook de F2 score, deze weegt de recall twee maal zwaarder en is dus interessant voor onze toepassing.

Uiteindelijk moet een model met een mAP@.5:.95 haalbaar zijn, maar onze dataset is wellicht niet groot genoeg. We stellen de ondergrens op mAP@.5:85%. Dit moet haalbaar zijn. We nemen telkens Precision en Recall mee en maximaliseren op recall, dit kan middels de F2 score.

De output is een werkend SOTA model. Ik heb een YOLO implementatie voorbij zien komen in het document "IV General Overview Deck - Use Cases.pdf" van IndiVillage. YOLO lijkt misschien een voor de hand liggend real-time oplossing. Maar we hoeven geen real-time oplossing, dus we kunnen binnen het YOLO systeem (nano, small, large, Xtra large) naar de beste oplossing kijken en ook erbuiten. Bij een custom model is het belangrijk dat we zowel het model als de gewichten werkend overhandigd krijgen.


Succesfactoren:
- Minimaal algemene score: mAP@.5:.85
- Gewenste algemene score: mAP@.5:.95
- Maximaliseer recall met respectievelijk hoge precision 




In [15]:
from pathlib import Path
from shutil import copy2

In [16]:
original_folder = Path(r"C:\Users\Gebruiker\Documents\Falcker\AI\data\OlieDetectie\met circle\oil_set_from_inspection_data\Review\missing files env\check-missing files")
check_dir = Path(r"C:\Users\Gebruiker\Documents\Falcker\AI\data\OlieDetectie\met circle\oil_set_from_inspection_data\Review\good")

original_images = [f for f in original_folder.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]
check_images = [f for f in check_dir.iterdir() if f.suffix.lower() in [".jpg", ".jpeg", ".png"]]

original_image_names = {img.stem for img in original_images}
check_image_names = {img.stem for img in check_images}


In [18]:
# Test if images exist in both folders
assert original_folder.exists(), f"Original folder does not exist: {original_folder}"
assert check_dir.exists(), f"Check folder does not exist: {check_dir}"

Now we're going to do the checking which images are missing

In [ ]:
missing_images = []
for img_name in original_image_names:
    for img_name2 in check_image_names:
        if img_name not in check_image_names:
            missing_images.append(img_name)
            # print(f"Missing image in check folder: {img_name}")

In [21]:
len(missing_images)

170

In [14]:
missing_dir = Path(r'C:\Users\Gebruiker\Documents\Falcker\AI\data\OlieDetectie\met circle\oil_set_from_inspection_data\Review\missing files env\missing')
for img_name in missing_images:
    copy2(original_folder / f"{img_name}.jpeg", missing_dir)